# 04: JDBC/ODBC Ingestion

**Exam objective (from Data Ingestion and Loading domain):** Use JDBC/ODBC or REST clients in notebooks to land data into cloud storage or directly into Unity-Catalog-governed tables, usually orchestrated and scheduled with Lakeflow Jobs.

**Free Edition note:** This exercise is conceptual. JDBC/ODBC ingestion requires an external database not available on Free Edition. Syntax examples are for reading only, not for hands-on execution.

**Conceptual position:** JDBC/ODBC ingestion is the "escape hatch" for external database sources when no managed connector exists. When a managed connector exists for the source, it's almost always preferable. JDBC/ODBC is what you reach for when you need direct database access from a notebook and can't or don't want to configure a managed pipeline.

## What JDBC/ODBC ingestion is

JDBC (Java Database Connectivity) and ODBC (Open Database Connectivity) 
are long-standing standards for connecting to relational databases from 
application code. Spark supports both natively — you can point a 
DataFrame read at a JDBC URL and it will connect to the source database, 
issue queries, and pull data back.

In Databricks, this shows up as:
- A PySpark read using `.format("jdbc")` with connection options.
- A SQL `SELECT` from a table registered as a foreign table pointing at 
  a JDBC source.

The mechanism is direct: your notebook makes a network connection to the 
source database, runs one or more SELECT statements, and pulls the 
result rows into a DataFrame or a Delta table.

## When JDBC/ODBC ingestion fits

The exam's decision framework treats JDBC/ODBC as the fallback option when better alternatives don't apply. Specifically:

**Use JDBC/ODBC when:**
- The source is a database with a JDBC or ODBC driver.
- No managed connector exists for the source database.
- You need one-off or ad-hoc data extraction from a database in a notebook.
- You're doing exploratory work where the overhead of setting up a managed pipeline isn't justified.

**Prefer other options when:**
- A managed Lakeflow Connect connector exists for the source (e.g., SQL Server, MySQL, PostgreSQL are all covered by managed connectors). Managed connectors handle CDC, incremental sync, schema evolution, and scheduling. JDBC/ODBC is a raw connection with none of that machinery.
- The data is already files in cloud storage — use Auto Loader or COPY INTO.
- You need change data capture. JDBC/ODBC pulls the state at query time; it doesn't track changes.

## The relationship to the ingestion framework

In the decision framework from exercise 03, JDBC/ODBC sits underneath Lakeflow Connect managed connectors. If a managed connector exists for the source, the framework routes you there first. JDBC/ODBC is what you use when either the connector doesn't exist or the use case doesn't warrant the overhead of setting one up.

The exam's phrasing — "usually orchestrated and scheduled with Lakeflow Jobs" — is a hint that JDBC/ODBC reads in notebooks are often the middle of a scheduled batch pipeline: the notebook connects, pulls data, writes to a Delta table, and Lakeflow Jobs handles the scheduling. This is the standard pattern when a managed connector 
isn't an option.

In [0]:
# Simple PySpark JDBC read 
jdbc_url = "jdbc:postgressl://source-db.example.com:5432/salesdb"

df = (spark.read
      .format("jdbc")
      .option("url", jdbc_url)
      .option("dbtable", "public.customers")
      .option("user", dbutils.secrets.get(scope="db_secrets", key="username"))
      .option("password", dbutils.secrets.get(scope="db_secrets", key="password_key"))
      .option("driver", "org.postgresql.Driver")
      .load()
)

df.write.saveAsTable("certprep.ingestion.customers")

**Key options for JDBC read:**

- `url`: JDBC connection string. Format is driver-specific but usually `jdbc:<db-type>://<host>:<port>/<database>`.
- `dbtable`: The table to read from, or a subquery in parentheses if you want to filter/transform on the source side. Both `dbtable` and `query` are supported; use one or the other.
- `user` / `password`: Credentials. Never hardcode these. Use Databricks secrets (`dbutils.secrets.get(...)`) instead.
- `driver`: The JDBC driver class. Databricks includes drivers for common databases (PostgreSQL, MySQL, SQL Server, Oracle, etc.) — you don't usually install them yourself, but you do need to name the right class.

In [0]:
# Simple PySpark JDBC read with a query
df = (spark.read
      .option("url", jdbc_url)
      .option("query", "SELECT customer_id, name, region FROM public.customers WHERE region = 'North'")
      .option("user", dbutils.secrets.get(scope="db_secrets", key="username"))
      .option("password", dbutils.secrets.get(scope="db_secrets", key="password_key"))
      .option("driver", "org.postgresql.Driver")
      .load()
)

In [0]:
# PySpark parallel reads with partitioning
df = (spark.read
      .option("url", jdbc_url)
      .option("dbtable", "public.orders")
      .option("user", dbutils.secrets.get(scope="db_secrets", key="username"))
      .option("password", dbutils.secrets.get(scope="db_secrets", key="password_key"))
      .option("driver", "org.postgresql.Driver")
      .option("partitionColumn", "order_id")
      .option("lowerBound", "1")
      .option("upperBound", "10000000")
      .option("numPartitions", "10")
      .load()
)

**Partitioned JDBC read options:**

- `partitionColumn`: A numeric column used to split the read into ranges. Must be numeric — string partition columns aren't supported directly.
- `lowerBound` / `upperBound`: The range of the partition column. Spark uses these to compute range boundaries for each partition.
- `numPartitions`: Number of parallel connections to open.

**How this works:** 
Spark divides the range `[lowerBound, upperBound]` into `numPartitions` equal-sized chunks and issues one query per chunk, each with a `WHERE partitionColumn BETWEEN low AND high` clause. Each runs on a separate executor in parallel.

**Important gotcha:** 
`lowerBound` and `upperBound` don't filter the data — they only control partition boundaries. Rows outside the range still come back (they just all end up in either the first or last partition). If you want to filter, use `query` or a WHERE clause in `dbtable`.

In [0]:
%sql
-- Register a connection to the source db
CREATE CONNECTION postgres_source
TYPE POSTGRESQL
OPTIONS (
    host        'source-db.example.com',
    port        '5432',
    user        secret('db_secrets', 'username'),
    password    secret('db_secrets', 'password_key')
); 

-- Create a foreign catalog that mirrors the source database
CREATE FOREIGN CATALOG postgres_salesdb
USING CONNECTION postgres_source
OPTIONS(database 'salesdb');

-- Now query as if it were a native table
SELECT * 
FROM postgres_salesdb.public.customers 
WHERE region = 'North';

**Foreign tables via Lakehouse Federation:**

Instead of pulling data with a JDBC read into a local Delta table, Lakehouse Federation lets you query the source database *in place* from Databricks. UC treats the external database as a foreign catalog; queries against it are translated and executed against the source.

**When this fits vs. a JDBC read into Delta:**

- **Foreign table (federation)**: When you need occasional, ad-hoc access and don't want a copy of the data. Query latency depends on the source database.
- **JDBC read into Delta**: When you need repeated fast reads over the data, or you need to join with other Databricks data at scale, or you need a stable snapshot. The Delta copy pays a one-time ingestion cost for repeatable, fast, colocated access.

Foreign tables are a newer capability and the exam may test whether you know they exist and what tradeoffs they make. But for straight "ingest this database's data" scenarios, the JDBC read into Delta pattern is still the default.

## Operational considerations for JDBC/ODBC ingestion

### 1. Credentials management

Never hardcode database credentials in notebook code. Two acceptable patterns:

**Databricks secrets (recommended):**
Store credentials in a secret scope, retrieve them at read time:
```python
.option("user", dbutils.secrets.get(scope="db_secrets", key="username"))
.option("password", dbutils.secrets.get(scope="db_secrets", key="password_key"))
```
Secrets are encrypted at rest, access-controlled, and don't appear in notebook output even if the notebook is exported.

**SQL secret() function:**
For SQL-based reads (foreign tables, connections):
```sql
user secret('db_secrets', 'username'),
password secret('db_secrets', 'password_key')
```

The exam will penalize scenarios where credentials are visible as string literals in code.

### 2. Network access

The Databricks cluster needs network access to the source database. This is not automatic:

- **VPC peering** or **private link** for databases in the same cloud region.
- **Firewall rules** allowing the cluster's outbound IP range on the source database's inbound port.
- **On-premises databases** typically require a VPN or Direct Connect   bridge.

This is often the blocker in real setups. A JDBC read that works from a developer's laptop but not from a Databricks cluster is almost always a network access issue, not a code issue.

### 3. Source database load

Every JDBC read consumes resources on the source database — connections, query planner capacity, IO. A JDBC read of a 100M-row production table with 20 parallel partitions is putting the equivalent of a large analytical workload on the source.

Mitigation patterns:

- **Read from a replica** rather than the primary. Most production databases have a read replica specifically for this kind of analytical access.
- **Schedule during off-peak hours.** Nightly ingestion at 2 AM instead of hourly during business hours.
- **Limit parallelism.** `numPartitions = 4` is often sufficient and much kinder to the source than `numPartitions = 32`.
- **Push predicates to the source.** Filter with `query` or a subquery in `dbtable` rather than pulling everything and filtering in Spark. Less data over the network, less scan on the source.

### 4. Incremental ingestion is your responsibility

JDBC/ODBC has no built-in incremental machinery. Every read is a fresh query against the source. If you want to only pull new rows since the last run, you have to implement it yourself:

**Watermark pattern:**
Track the maximum value of a monotonically-increasing column (an updated_at timestamp, an ID) from the previous run, and use it as a filter on the next read.

```python
last_watermark = spark.sql(
    "SELECT MAX(updated_at) FROM certprep.ingestion.customers"
).collect()[0][0]

query = f"""
  SELECT * FROM public.customers 
  WHERE updated_at > '{last_watermark}'
"""
```

This is exactly the pattern that Auto Loader and COPY INTO handle for you automatically. The exam objective's phrase "usually orchestrated and scheduled with Lakeflow Jobs" is a hint that JDBC/ODBC pipelines require external orchestration for incremental behavior — the notebook does the read, Lakeflow Jobs handles the schedule and the watermark state.

**This is why managed connectors are preferred when they exist.** 
CDC and incremental sync are their whole point. Rolling your own watermarking is fine for occasional use, but it's fragile — watermark columns need to be reliable, late-arriving rows can be missed, and out-of-order updates are hard to handle.

## Self Check 

1. You review a colleague's notebook that reads data from a PostgreSQL database via JDBC. The read works but takes 90 minutes on a table that shouldn't take that long. Looking at the code, you notice there's no `partitionColumn`, `lowerBound`, or `upperBound` specified. What's happening, and what's the fix?

2. A team is setting up incremental ingestion from a MySQL database into Delta. They're considering two approaches: (a) a managed Lakeflow Connect connector for MySQL, or (b) a JDBC read in a scheduled notebook with a watermark pattern. What's the tradeoff, and which would you recommend by default?

3. Explain what happens in this JDBC read, given that the `order_id` column in the source actually ranges from 1 to 50,000,000:

```python
   df = (spark.read
       .format("jdbc")
       .option("url", jdbc_url)
       .option("dbtable", "public.orders")
       .option("partitionColumn", "order_id")
       .option("lowerBound", "1")
       .option("upperBound", "1000000")
       .option("numPartitions", "10")
       .load()
   )
```

4. A team is doing occasional ad-hoc analysis against a Snowflake data warehouse from Databricks. They don't want a local copy of the data — it changes constantly and they don't want to manage a sync. What's the appropriate pattern, and how does it differ from a JDBC read into a Delta table?

5. You see this line in a notebook.  What's wrong, and what's the correct pattern?
```python
   .option("password", "S3cur3P@ssw0rd!")
```

6. A JDBC read is failing with a network timeout error. The same connection works from a developer's local machine using the same credentials. The Databricks cluster is running in the same cloud region as the source database. What's likely wrong?

## Self Check Answers

1. The JDBC read is performing the entire read via a single executor. The single query is likely intensive on the source platform and needs to be broken up via partitioning. To do this, we first need a numerical column to partition on, such as an ID column, to set for `partitionColumn`. Next, we need to set a boundary for the partitioning, via `lowerBound` and `upperBound`. This may require looking at the source data for the partition column itself to get an idea of the numeric values therein to determine an effective range. Lastly, we'd need to use `numPartitions` to set the actual number of partitions to split the data across. We'd have to take into account how many executors we have available to us in the cluster as well as the source database load when determining how many partitions to use. Too many parallel processes running on the source side can overwhelm the source system or hit connection limits. If our boundary range is set efficiently, we can spread the read's workload accross multiple executors in parallel with even distribution, save the data outside of the lower and upper bounds being read by the executors that read the first and last partitions.
2. By default I would recommend the Lakeflow Connect connector for MySQL. Lakeflow's offering already manages incremental loads with CDC and incremental sync, whereas a JDBC read will require you to manage the set up for incremental loading. However, if this ingestion is more on the side of a one-off, then it may not warrant the effort of setting up a connector. In that case, the JDBC connection is suitable.
3. This code would result in a large data skew. The first 9 partitions would contain rows with the order_id values 1-1000000, while the 10th partition would contain 1000001-50000000.
4. THe traditional pattern is using Lakehouse Federation to create a "foreign catalog" via a connection to the source. This function allows you to use the foreign catalog to query the source table as if it were a native table without ever storing a copy of the actual data. JDBC read into a Delta table does not do this, as it is a literal read into Delta, thus you are storing a copy of the data. 
5. The issue is that the password is a hard-coded string, which should never be the case for obvious security concerns. The password should be stored as a secret with a scope and key to access the secret. Then replace the string with `dbutils.secrets.get(scope="scope", key="key")` for example.
6. The most likely issue is network access restrictions. The cluster needs to be granted access to the source platform/databricks in order to be able to query it. Fixes include VPC peering between Databricks workspace VNet/VPC and the database network, a private link or endpoint (same effect via a managed service), updating firewall rules on the source database to allow the cluster's IP range, and NAT gateway or secure cluster connectivity configurations depending on the workspace setup.